# Knee MRI Abnormality Detection — Phase 3: Slice Feature Extraction & Offline Caching

This dedicated notebook extracts and caches 2D slice feature vectors across all multi-slice MRI volumes in the RSNA dataset.

### Key Highlights:
1. **Multiple Vision Encoders:** Choose between **ResNet-50** (`resnet50`), **DINOv2** (`dinov2` / `dinov2_small` / `dinov2_base`), or **DINO-v3** (`dino_v3` / `dinov3_small` / `dinov3_base`).
2. **Anatomical Ordering:** DICOM slices are sorted along the true physical z-axis (`ImagePositionPatient[2]`).
3. **Parallel Multi-Core Loading:** Uses multi-threaded decoding (`ThreadPoolExecutor`) across CPU cores to saturate GPU compute.
4. **Atomic & Resumable Caching:** Saves each series as `[num_slices, feature_dim]` numpy array. Skips already cached series if restarted.
5. **Dataset Export / Persistence:** Automatically packages outputs so they can be saved as a persistent Kaggle Dataset and attached to downstream training notebooks.


In [ ]:
# --- Cell 1: Git Sync & Dependency Installation ---
import os, sys, subprocess
from pathlib import Path

# Handle optional private GitHub PAT via Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_PAT")
    REPO_URL = f"https://{token}@github.com/Prashanf/Rsna_KneeAbnormality_Detection.git"
except Exception:
    REPO_URL = "https://github.com/Prashanf/Rsna_KneeAbnormality_Detection.git"

REPO_DIR = "/kaggle/working/repo"

if os.path.exists("/kaggle/working"):
    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo from {REPO_URL} into {REPO_DIR}...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print(f"Pulling latest changes in {REPO_DIR}...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    print(f"✓ Repository synced at commit: {commit}")

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    local_root = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
    print(f"Running in local workspace: {local_root}")
    if local_root not in sys.path:
        sys.path.insert(0, local_root)

# Ensure medical imaging & vision dependencies are installed
try:
    import pydicom, pylibjpeg, timm
except ImportError:
    print("Installing DICOM decoders and timm...")
    !pip install -q pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg timm


In [ ]:
# --- Cell 2: Purge Cache & Resolve Dataset Paths ---
for mod in list(sys.modules):
    if mod.startswith("src.") or mod.startswith("environment."):
        del sys.modules[mod]

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from environment.imports import *

def resolve_paths():
    kaggle_matches = list(Path("/kaggle/input").glob("*knee*")) + list(Path("/kaggle/input").glob("*rsna*"))
    if kaggle_matches:
        raw_dir = kaggle_matches[0]
        processed_dir = Path("/kaggle/working/data/processed")
        return raw_dir, processed_dir
    for candidate in [Path("data/raw"), Path("../data/raw")]:
        if candidate.exists():
            return candidate.resolve(), candidate.parent.resolve() / "processed"
    return Path("data/raw"), Path("data/processed")

RAW_DIR, PROCESSED_DIR = resolve_paths()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"RAW_DIR:       {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")
print(f"Compute Device: {device} ({torch.cuda.get_device_name(0) if device == "cuda" else "CPU"})")


In [ ]:
# --- Cell 3: Select & Initialize Vision Encoder ---
from src.encoders import get_encoder

# ==========================================================================
# ENCODER SELECTION:
# - "resnet50":      2048-dim features. Industry-standard CNN, fast (~40-50 slices/sec)
# - "dinov2":        384-dim features. Meta self-supervised ViT-Small (rich patch features)
# - "dinov2_base":   768-dim features. Meta self-supervised ViT-Base (higher capacity)
# - "dino_v3":       384-dim features. Meta DINOv3 ViT architecture
# - "dinov3_base":   768-dim features. Meta DINOv3 ViT-Base
# ==========================================================================
ENCODER_CHOICE = "resnet50"  # Select: "resnet50", "dinov2", "dino_v3", "dinov2_base", "dinov3_base"

# Execution Settings
BATCH_SIZE = 32         # Slices per GPU forward pass (32 or 64 recommended)
NUM_WORKERS = 4        # Parallel CPU threads for DICOM decompression (saturates GPU)
LIMIT_SERIES = None    # Set to None for ALL series, or e.g. 50 for a fast verification trial
OVERWRITE = False      # Skips already completed .npy series (resumable)

print(f"Instantiating encoder '{ENCODER_CHOICE}' on {device}...")
encoder = get_encoder(name=ENCODER_CHOICE, pretrained=True, device=device)

# Quick forward smoke test with dummy tensor
dummy_input = torch.zeros((2, 3, encoder.img_size, encoder.img_size), device=device)
with torch.no_grad():
    dummy_out = encoder.encode(dummy_input)

print("✓ Encoder initialized successfully:")
print(f"  Name:        {encoder.name}")
print(f"  Feature Dim: {encoder.feature_dim}")
print(f"  Input Size:  {encoder.img_size}x{encoder.img_size}")
print(f"  Smoke Test:  Input {list(dummy_input.shape)} -> Output {list(dummy_out.shape)}")


In [ ]:
# --- Cell 4: Check Existing Embeddings Cache ---
embeddings_dir = PROCESSED_DIR / "embeddings" / ENCODER_CHOICE
embeddings_dir.mkdir(parents=True, exist_ok=True)

# Load series list
series_index_path = PROCESSED_DIR / "series_index.parquet"
if series_index_path.exists():
    series_df = pd.read_parquet(series_index_path)
else:
    series_df = pd.read_csv(RAW_DIR / "train_series.csv")

total_cohort_series = len(series_df)
cached_npy_files = list(embeddings_dir.glob("*/*.npy"))
cached_count = len(cached_npy_files)

print(f"Total Series in Dataset:       {total_cohort_series}")
print(f"Already Cached on Disk:       {cached_count} ({cached_count/max(total_cohort_series, 1)*100:.1f}%)")
print(f"Remaining Series to Embed:     {max(0, total_cohort_series - cached_count)}")
print("Any rerun will safely resume from where it left off without duplicating work.")


In [ ]:
# --- Cell 5: Run Parallel Slice Embedding Pipeline ---
from src.embed_slices import embed_all_series

print(f"Starting slice feature extraction with '{ENCODER_CHOICE}'...")
print(f"Target directory: {embeddings_dir}")
if LIMIT_SERIES:
    print(f"Limited mode enabled: processing first {LIMIT_SERIES} series only.")

start_time = time.time()
manifest = embed_all_series(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    encoder_name=ENCODER_CHOICE,
    batch_size=BATCH_SIZE,
    overwrite=OVERWRITE,
    limit=LIMIT_SERIES,
    device="auto",
    num_workers=NUM_WORKERS,
)
elapsed = time.time() - start_time

print("
" + "=" * 65)
print(f"      PHASE 3 SLICE EMBEDDING COMPLETE ({elapsed/60:.1f} minutes)")
print("=" * 65)
print(f"  Encoder:          {manifest.get('encoder_name')}")
print(f"  Feature Dim:      {manifest.get('feature_dim')}")
print(f"  Total Series:     {manifest.get('total_series')}")
print(f"  Newly Embedded:   {manifest.get('embedded_count')}")
print(f"  Skipped (Cached): {manifest.get('skipped_count')}")
print(f"  Failed:           {manifest.get('failed_count')}")
print("=" * 65)


In [ ]:
# --- Cell 6: Validate & Visualize Generated Embeddings ---
embedded_files = list(embeddings_dir.glob("*/*.npy"))
print(f"Total cached .npy series files: {len(embedded_files)}")

if embedded_files:
    sample_path = embedded_files[0]
    arr = np.load(str(sample_path))
    
    print(f"
Sample Series Verification ({sample_path.parent.name} / {sample_path.stem}):")
    print(f"  Matrix Shape:   {arr.shape}  [slices={arr.shape[0]}, feature_dim={arr.shape[1]}]")
    print(f"  Value Range:    min={arr.min():.4f}, max={arr.max():.4f}, mean={arr.mean():.4f}")
    print(f"  Integrity:      has_nan={np.isnan(arr).any()}, has_inf={np.isinf(arr).any()}")
    
    # Diagnostic Plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Feature activation heatmap across anatomical slices
    dims_to_show = min(120, arr.shape[1])
    im = axes[0].imshow(arr[:, :dims_to_show], aspect="auto", cmap="viridis")
    axes[0].set_title(f"Slice Feature Profile (First {dims_to_show} Dims)")
    axes[0].set_xlabel("Feature Dimension")
    axes[0].set_ylabel("Slice Index (Anatomical Z-Order)")
    plt.colorbar(im, ax=axes[0], label="Activation")
    
    # 2. Slice count distribution across processed series
    slice_counts = []
    for f in embedded_files[:200]:
        try:
            m = np.load(str(f), mmap_mode="r")
            slice_counts.append(m.shape[0])
        except Exception:
            pass
            
    axes[1].hist(slice_counts, bins=25, color="#1f77b4", edgecolor="black", alpha=0.75)
    axes[1].set_title(f"Slices per Series Distribution (Sampled N={len(slice_counts)})")
    axes[1].set_xlabel("Number of Slices")
    axes[1].set_ylabel("Count")
    axes[1].grid(axis="y", linestyle="--", alpha=0.6)
    
    plt.tight_layout()
    plt.show()
else:
    print("Notice: No embeddings found to visualize.")


In [ ]:
# --- Cell 7: Persist Embeddings & Kaggle Dataset Export ---
import shutil
import tarfile

# Summary of directory sizes
all_files = list(embeddings_dir.glob("*/*.npy"))
total_bytes = sum(f.stat().st_size for f in all_files)
total_mb = total_bytes / (1024 * 1024)
total_gb = total_mb / 1024

print("=" * 65)
print("          EMBEDDINGS PERSISTENCE & EXPORT SUMMARY")
print("=" * 65)
print(f"  Encoder:             {ENCODER_CHOICE}")
print(f"  Total Files:         {len(all_files)} .npy files")
print(f"  Total Disk Footprint: {total_mb:.1f} MB ({total_gb:.2f} GB)")
print(f"  Location:            {embeddings_dir}")
print("=" * 65)

# Option: Create compressed tarball for easy single-file download or dataset export
CREATE_TARBALL = False  # Set to True if you want a single downloadable .tar.gz archive

if CREATE_TARBALL and len(all_files) > 0:
    archive_name = f"/kaggle/working/knee_mri_embeddings_{ENCODER_CHOICE}.tar.gz"
    print(f"Creating compressed archive {archive_name}...")
    with tarfile.open(archive_name, "w:gz") as tar:
        tar.add(embeddings_dir, arcname=f"embeddings/{ENCODER_CHOICE}")
    print(f"✓ Archive created: {archive_name} ({Path(archive_name).stat().st_size / (1024*1024):.1f} MB)")

print("
✓ HOW TO REUSE THESE EMBEDDINGS IN FUTURE TRAINING NOTEBOOKS:")
print("1. In the top-right corner of Kaggle, click 'Save Version' -> 'Save & Run All (Commit)'.")
print("2. After completion, in your training notebook (04_train.ipynb), click '+ Add Input'.")
print("3. Select 'Your Work' -> '03_slice_embeddings' (the output of this notebook).")
print("4. All embeddings will be instantly accessible at /kaggle/input/03-slice-embeddings/data/processed/embeddings/!")
